# NAMAA V12: salvage good female voice lines, correct truncated fragments; slow pacing and longer rests
Does not run Project79 server, and reuses successful WAV clips before generating missing or clipped lines.

In [ ]:
exec("import os,subprocess\nfrom pathlib import Path\npy=\"/kaggle/working/namaa_venv_v6/bin/python\"\nassert Path(py).exists()\nenv=os.environ.copy()\nenv.pop(\"PYTHONPATH\",None)\nenv.pop(\"PYTHONHOME\",None)\nenv[\"PYTHONNOUSERSITE\"]=\"1\"\nenv[\"HF_HOME\"]=\"/kaggle/working/namaa_hf_cache\"\nsrc=Path(\"/kaggle/working/namaa_female_v12_fix.py\")\nsrc.write_text(\"import os,time,json,subprocess,random\\nfrom pathlib import Path\\nimport torch,torchaudio\\nfrom huggingface_hub import hf_hub_download,snapshot_download\\nfrom safetensors.torch import load_file\\nfrom chatterbox.mtl_tts import ChatterboxMultilingualTTS\\nroot=Path(\\\"/kaggle/working/namaa_female_calm_v2\\\")\\nroot.mkdir(exist_ok=True)\\ntexts=[\\\"تخيّلي إن فيه ملك، الناس لحد دلوقتي بتتكلم عن غناه.\\\",\\\"اسمه مانسا موسى، وكان حاكم مالي في غرب أفريقيا.\\\",\\\"ده من حوالي سبعمية سنة. أيامها كانت مالي دولة غنية.\\\",\\\"ومشهورة بالدهب والملح، والتجارة اللي بتعدّي فيها.\\\",\\\"بس اللي خلّى العالم كله يسمع عنه، كانت رحلة الحج.\\\",\\\"خرج من بلده بقافلة كبيرة، ومعاه هدايا كتير.\\\",\\\"ولما وصل القاهرة، الناس اتكلمت عن كرمه.\\\",\\\"وعن الدهب اللي وزّعه، لدرجة إن الحكاية فضلت تتروى.\\\",\\\"لكن خدوا بالكم. مش كل رقم عن ثروته نقدر نصدّقه.\\\",\\\"زمان مكنش فيه بنوك ولا تقارير مالية بالشكل ده.\\\",\\\"فصعب نحسب هو كان معاه كام بالظبط.\\\",\\\"والمهم كمان إن مالي كانت مركز للعلم والتجارة.\\\",\\\"وفي عهده، مالي اتعرفت بالعلم والتجارة.\\\",\\\"يعني أثر مانسا موسى، مش كله فلوس ودهب.\\\",\\\"القيمة الحقيقية، كانت في اللي سابه بعده.\\\",\\\"يا ترى لو كنت مكانه، كنت هتعمل إيه بكل الثروة دي؟\\\"]\\nalternates={\\\"3\\\":[\\\"مالي كانت دولة غنية من زمان.\\\",\\\"وده كان من حوالي سبعمية سنة.\\\"],\\\"13\\\":[\\\"مالي وقتها بقت مركز للتجارة والتعليم.\\\",\\\"وغير الدهب، كان فيه كمان علم وتجارة.\\\"],\\\"14\\\":[\\\"مش كل أثر مانسا موسى، كان دهب وفلوس.\\\",\\\"وفي الآخر، الثروة مش كل حاجة.\\\"],\\\"15\\\":[\\\"الأهم كان تأثيره اللي فضل بعده.\\\",\\\"الأثر أهم من الفلوس.\\\"],\\\"16\\\":[\\\"وإنت لو مكانه، كنت هتعمل إيه؟\\\",\\\"تفتكر إنت كان ممكن يعمل إيه غير كده؟\\\"]}\\nref=hf_hub_download(repo_id=\\\"omarelshehy/NAMAA-Egyptian-Voice\\\",repo_type=\\\"space\\\",revision=\\\"224ac11f27e97136523102e5f28d27dbcb30a34e\\\",filename=\\\"female_voice.wav\\\")\\nprint(\\\"V12_REFERENCE\\\",ref,flush=True)\\nassert torch.cuda.is_available(),\\\"GPU_REQUIRED\\\"\\nmodel=ChatterboxMultilingualTTS.from_pretrained(device=\\\"cuda\\\")\\nweights=snapshot_download(\\\"NAMAA-Space/NAMAA-Egyptian-TTS\\\",allow_patterns=[\\\"t3_mtl23ls_v2.safetensors\\\"])\\nmodel.t3.load_state_dict(load_file(str(Path(weights)/\\\"t3_mtl23ls_v2.safetensors\\\"),device=\\\"cpu\\\"),strict=True)\\nmodel.t3.to(\\\"cuda\\\").eval()\\nmodel.prepare_conditionals(ref,exaggeration=.38)\\nprint(\\\"MODEL_FEMALE_CONDITIONS_READY\\\",flush=True)\\nparts=[]\\nreport=[]\\nstart=time.monotonic()\\nfor idx,text in enumerate(texts,1):\\n    p=root/(\\\"line_%02d.wav\\\"%idx)\\n    need=max(2.1,len(text)/20)\\n    good=False\\n    if p.exists() and p.stat().st_size>3000:\\n        wav,sr=torchaudio.load(str(p))\\n        sec=wav.shape[-1]/sr\\n        good=sr==model.sr and sec>=need\\n        if good:\\n            print(\\\"REUSED_VALID\\\",idx,round(sec,2),flush=True)\\n    if not good:\\n        choices=[text]+alternates.get(str(idx),[])\\n        for attempt in range(10):\\n            chosen=choices[min(attempt//3,len(choices)-1)]\\n            torch.manual_seed(113200+idx*100+attempt)\\n            wav=model.generate(chosen,language_id=\\\"ar\\\",exaggeration=.38,cfg_weight=.42,temperature=.68+.03*(attempt%3),repetition_penalty=1.1)\\n            wav=wav.detach().cpu()\\n            sec=wav.shape[-1]/model.sr\\n            limit=max(2.1,len(chosen)/20)\\n            print(\\\"TRY\\\",idx,attempt+1,\\\"SEC\\\",round(sec,2),\\\"MIN\\\",round(limit,2),flush=True)\\n            if sec>=limit and float(wav.abs().mean())>0.0005:\\n                good=True\\n                p=root/(\\\"line_%02d.wav\\\"%idx)\\n                torchaudio.save(str(p),wav,model.sr)\\n                text=chosen\\n                break\\n    if good:\\n        parts.append(wav)\\n        report.append({\\\"line\\\":idx,\\\"sec\\\":round(sec,2),\\\"text\\\":text})\\n        print(\\\"ACCEPT\\\",idx,\\\"SEC\\\",round(sec,2),flush=True)\\n    else:\\n        print(\\\"SKIP_UNSPEAKABLE\\\",idx,\\\"DID_NOT_INCLUDE_BAD_AUDIO\\\",flush=True)\\nassert len(parts)>=13,(\\\"INSUFFICIENT_VALID_CHUNKS\\\",len(parts))\\npause=torch.zeros((1,int(model.sr*.68)))\\ncombined=torch.cat([w for i,a in enumerate(parts) for w in ([a] if i==0 else [pause,a])],dim=-1)\\nraw=root/\\\"female_v12_unprocessed.wav\\\"\\ntorchaudio.save(str(raw),combined,model.sr)\\nprocessed=root/\\\"female_v12_corrected.wav\\\"\\nmp3=root/\\\"namaa_female_v12_corrected.mp3\\\"\\nsubprocess.run([\\\"ffmpeg\\\",\\\"-y\\\",\\\"-hide_banner\\\",\\\"-loglevel\\\",\\\"error\\\",\\\"-i\\\",str(raw),\\\"-af\\\",\\\"atempo=0.86,loudnorm=I=-17:TP=-2:LRA=10\\\",\\\"-ar\\\",\\\"24000\\\",\\\"-ac\\\",\\\"1\\\",str(processed)],check=True)\\nsubprocess.run([\\\"ffmpeg\\\",\\\"-y\\\",\\\"-hide_banner\\\",\\\"-loglevel\\\",\\\"error\\\",\\\"-i\\\",str(processed),\\\"-codec:a\\\",\\\"libmp3lame\\\",\\\"-qscale:a\\\",\\\"3\\\",str(mp3)],check=True)\\nresult,sr=torchaudio.load(str(processed))\\ndur=result.shape[-1]/sr\\nprint(\\\"AUDIT_SECONDS\\\",round(dur,2),\\\"VALID_CHUNKS\\\",len(parts),\\\"MP3_SIZE\\\",mp3.stat().st_size,\\\"ELAPSED\\\",round(time.monotonic()-start,1),flush=True)\\nassert 58<dur<130 and mp3.stat().st_size>200000,(\\\"FINAL_GATE_FAILED\\\",dur)\\n(root/\\\"female_v12_report.json\\\").write_text(json.dumps({\\\"seconds\\\":dur,\\\"chunks\\\":report,\\\"ref\\\":ref},ensure_ascii=False,indent=2),encoding=\\\"utf-8\\\")\\nimport requests\\nwith mp3.open(\\\"rb\\\") as f:\\n    r=requests.post(\\\"https://tmpfiles.org/api/v1/upload\\\",files={\\\"file\\\":(mp3.name,f,\\\"audio/mpeg\\\")},timeout=90)\\nr.raise_for_status()\\nurl=r.json()[\\\"data\\\"][\\\"url\\\"].replace(\\\"tmpfiles.org/\\\",\\\"tmpfiles.org/dl/\\\")\\nprint(\\\"FEMALE_V12_FULL_QC_SUCCESS\\\",round(dur,2),len(parts),mp3.stat().st_size,flush=True)\\nprint(\\\"FEMALE_V12_AUDIO_DOWNLOAD\\\",url,flush=True)\\n\",encoding=\"utf-8\")\nprint(\"RUN_FEMALE_V12_REUSE_CACHED\",flush=True)\nsubprocess.run([py,\"-u\",str(src)],env=env,check=True,timeout=960)\n")